# Validation

## Setup

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

%load_ext autoreload
%autoreload 2

import keras

## Upload LUT & NN

In [ ]:
from src.lut import LUT
from src.nn_validator import NN_validator, LUT_validator

LUT_PATH = "models/validation_luts/lut_W_0.85.nc"
MODEL_NAME = "3L_64U"
#NN_PATH = f"models/nn/W_all/grid_search/W_all_{MODEL_NAME}_relu_linear/W_all_{MODEL_NAME}_relu_linear.keras"
NN_PATH = f"models/nn/W_all/grid_search/W_all_{MODEL_NAME}_relu_linear_cos_decay/W_all_{MODEL_NAME}_relu_linear_cos_decay.keras"

my_lut = LUT(LUT_PATH)
my_lut.load_data()

train_lut = LUT("models/luts/W_all/lut_W_all.nc")
train_lut.load_data()

nn_validator = NN_validator(NN_PATH, my_lut) 
lut_validator = LUT_validator(train_lut, my_lut) 

## Metrics & Plots

**Notation**: $y_i$ target value (LUT), $\hat y_i$ network prediction, $N$ number of samples,
$\bar y = \frac{1}{N}\sum_{i=1}^{N} y_i$ mean of the targets.

**Mean Squared Error** $:=\mathrm{MSE} = \frac{1}{N}\sum_{i=1}^{N} \left(y_i - \hat y_i\right)^2$

**Root Mean Squared Error** $:=\mathrm{RMSE} = \sqrt{\mathrm{MSE}} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} \left(y_i - \hat y_i\right)^2}$

**Maximum absolute error** (worst case) $:=\max|\mathrm{err}| = \max_{i=1,\dots,N} \left|y_i - \hat y_i\right|$

**Coefficient of determination** $:=R^2 = 1 - \frac{\sum_{i=1}^{N}\left(y_i - \hat y_i\right)^2}{\sum_{i=1}^{N}\left(y_i - \bar y\right)^2}$

**Normalized fit** $:=\mathrm{FIT}\,[\%] = 100\left(1 - \frac{\lVert y - \hat y \rVert_2}{\lVert y - \bar y \rVert_2}\right)
= 100\left(1 - \sqrt{\frac{\sum_{i=1}^{N}\left(y_i - \hat y_i\right)^2}{\sum_{i=1}^{N}\left(y_i - \bar y\right)^2}}\right)$

In [ ]:
from src.nn_model_visualizer import NN_models_visualizer

SAVE_PATH_PLOTS = os.path.join(os.path.dirname(NN_PATH), "plots", "validation")

visualizer = NN_models_visualizer([nn_validator, lut_validator])
visualizer.print_table(["mse", "rmse", "max_error", "r2", "fit_pct", "non_finite"])
visualizer.plot_parity(show=True,
                       #save_path=SAVE_PATH_PLOTS
                       )